In [2]:
# 1. Install libraries
!pip install transformers datasets scikit-learn matplotlib seaborn

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 5.9 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/10.4 MB 6.7 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.4 MB 6.7 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.4 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 6.3/10.4 MB 6.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.4 MB 6.7 MB/s eta 0:00:01
   ------------------------------------ --- 9.4/10.4 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 6.7 MB/s  0:00:01
   ---------------------------------------- 0.0/553.3 kB ? eta -:--:--
   ---------------------------------------- 553.3/553.3 kB 5.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------------------ --------------------- 1.3/2.9 MB 7.6 MB/s eta 0:00:01
   --------------------

In [ ]:


import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm
import os

# 2. Load your dataset
df = pd.read_csv('/kaggle/input/multilingual-meme-datasets/final_datasets.csv')  # Replace with your path
df = df[['translated_text', 'label']].dropna()

# 3. Prepare Train and Validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['translated_text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2,
    random_state=42
)

# 4. Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# 5. Create Dataset class
class MemeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = MemeDataset(train_encodings, train_labels)
val_dataset = MemeDataset(val_encodings, val_labels)

# 6. Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# 7. Set up optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# 8. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# 9. Training Loop with Loss Tracking
epochs = 5
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    loop = tqdm(train_loader, leave=True)

    for batch in loop:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()

        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    #  Validation Phase
    model.eval()
    total_val_loss = 0
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_val_loss += loss.item()

            logits = outputs.logits
            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())




    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    acc = accuracy_score(true_labels, preds)
    val_accuracies.append(acc)

    print(f"\n Epoch {epoch+1} finished:")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val Accuracy: {acc:.4f}")

# 10. Save model and tokenizer
save_path = './saved_bert_model'
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\n Model and tokenizer saved at", save_path)

# 11.  Confusion Matrix and Classification Report
cm = confusion_matrix(true_labels, preds)
report = classification_report(true_labels, preds, digits=4)

print("\n Classification Report:")
print(report)

# 12.  Plot Loss and Accuracy Graphs
epochs_range = range(1, epochs+1)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, val_accuracies, label='Validation Accuracy', color='green')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# 13.  Confusion matrix plot
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


c:\Users\moham\anaconda3\Lib\site-packages\numpy\__init__.py:109: UserWarning: mkl-service package failed to import, therefore Intel(R) MKL initialization ensuring its correct out-of-the box operation under condition when Gnu OpenMP had already been loaded by Python process is not assured. Please install mkl-service package, see http://github.com/IntelPython/mkl-service
  from . import _distributor_init


ModuleNotFoundError: No module named 'transformers'